# 02 - Build Retrieval Index v3

Builds dense FAISS and BM25 indexes from the normalized official-law corpus. This notebook uses `retrieval_text` for embeddings and saves metadata needed for citation-aware retrieval.

In [ ]:
from pathlib import Path
import json
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
corpus_path = DRIVE_ROOT / config['main_law_corpus_csv']
index_root = DRIVE_ROOT / config.get('official_index_root', 'indexes/official_law_v3')

if not corpus_path.exists():
    raise FileNotFoundError(f'Missing normalized corpus: {corpus_path}. Run notebook 01 first.')

corpus_path, index_root

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = {
    'sentence_transformers': 'sentence-transformers',
    'faiss': 'faiss-cpu',
    'rank_bm25': 'rank-bm25',
    'tqdm': 'tqdm',
}

missing_packages = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All retrieval dependencies are already installed.')

In [ ]:
import torch

embedding_model = config['embedding_models']['default']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32 if device == 'cuda' else 8

print('Embedding model:', embedding_model)
print('Device:', device)
print('Batch size:', batch_size)

In [ ]:
from src.build_index import build_indexes

manifest = build_indexes(
    corpus_path=corpus_path,
    index_root=index_root,
    embedding_model=embedding_model,
    text_field=config['retrieval_text_field'],
    batch_size=batch_size,
    device=device,
    build_dense=True,
    build_bm25=True,
)

manifest

In [ ]:
from src.retrieval import RetrievalEngine

engine = RetrievalEngine(index_root=index_root, device=device)
query = 'Kiraya veren hangi durumlarda tahliye davası açabilir?'
results = engine.hybrid_search(
    query,
    top_k=5,
    candidate_k=config['retrieval_defaults']['top_k_retrieval'],
    dense_weight=config['retrieval_defaults']['dense_weight'],
    bm25_weight=config['retrieval_defaults']['bm25_weight'],
)

[(r['citation_label'], r['article_key'], round(r['score'], 4)) for r in results]

Expected outputs under `indexes/official_law_v3/`:

- `dense/faiss.index`
- `dense/embeddings.npy`
- `bm25/bm25.pkl`
- `metadata/metadata.jsonl`
- `index_manifest.json`